# Notebook 10 — Debug Workflow

This notebook is a diagnostic companion.  When a simulation fails — or
gives wrong results — work through the stages below in order.  Each stage
narrows the root cause.

## Stages

| # | Stage | What it catches |
|---|-------|----------------|
| 0 | [Pre-flight: zombies and ports](#0) | Port conflicts, stale processes |
| 1 | [Config validation](#1) | Missing keys, wrong paths |
| 2 | [Geometry import verification](#2) | CDB element type, node CS, manifold |
| 3 | [Mesh quality gate](#3) | Inverted elements, high aspect ratio |
| 4 | [BC sanity check](#4) | Rigid body modes, wrong sign convention |
| 5 | [Solver diagnostic run](#5) | Residual divergence, substep failure |
| 6 | [Results sanity check](#6) | Wrong magnitude, wrong sign, NaN values |
| 7 | [Port / process audit](#7) | Zombie cleanup, license hang |
| 8 | [HFSS-specific checklist](#8) | Stale project, non-manifold STL |

---

<a id='0'></a>
## Stage 0 — Pre-flight: Zombies and Ports

**Run this EVERY TIME before launching MAPDL or HFSS.**

In [ ]:
import sys; sys.path.insert(0, '..')
from ams.resources.manager import kill_ansys_zombies, check_ports, ResourceManager

print('=== STEP 0a: Kill ANSYS zombie processes ===')
killed = kill_ansys_zombies(dry_run=False, include_aedt=True, verbose=True)

print(f'\n=== STEP 0b: Port status ===')
ports = check_ports()
for port, occupied in ports.items():
    status = 'OCCUPIED [!] -- something is listening' if occupied else 'free'
    print(f'  Port {port}: {status}')

occupied_ports = [p for p, s in ports.items() if s]
if occupied_ports:
    print(f'\n[!] Ports still occupied: {occupied_ports}')
    print('    These may block launch_mapdl().  Kill the owning processes manually')
    print('    or change the port in config.yaml: mapdl.port = 50053')
else:
    print('\n[OK] All ports free -- safe to launch')

In [ ]:
# STEP 0c: System resource check
rm   = ResourceManager()
info = rm.system_info()
print(f'Physical CPUs  : {info.physical_cpus}')
print(f'Total RAM      : {info.total_ram_mb:,} MB')
print(f'Available RAM  : {info.available_ram_mb:,} MB')
print(f'ANSYS version  : {info.ansys_version or "not detected"}')

if info.ansys_version is None:
    print('\n[!] ANSYS binary not detected at default paths.')
    print('    Set mapdl.custom_bin in config.yaml to the absolute path of ansysXXX.exe')

<a id='1'></a>
## Stage 1 — Config Validation

Check that `config.yaml` has all required sections and that file paths exist.

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('..') / 'config.yaml'
with open(cfg_path, encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

# Required top-level keys
required = ['problem', 'mapdl', 'geometry', 'elements', 'materials',
            'bcs', 'solver', 'resources', 'output']
missing = [k for k in required if k not in cfg]
if missing:
    print(f'[FAIL] Missing config sections: {missing}')
else:
    print('[OK] All required config sections present')

# Check geometry path if source == cdb
geom_src = cfg['geometry']['source']
if geom_src == 'cdb':
    cdb = cfg['geometry'].get('cdb_path')
    if not cdb:
        print('[FAIL] geometry.source == cdb but geometry.cdb_path is null')
    elif not Path(cdb).exists():
        print(f'[FAIL] CDB file not found: {cdb}')
    else:
        size_kb = Path(cdb).stat().st_size / 1024
        print(f'[OK] CDB file: {Path(cdb).name}  ({size_kb:.0f} KB)')
else:
    print(f'[OK] geometry.source = {geom_src} (no file path check needed)')

# Check output dir is writable
out_dir = Path(cfg['output']['dir'])
out_dir.mkdir(parents=True, exist_ok=True)
test_file = out_dir / '.write_test'
try:
    test_file.write_text('test', encoding='utf-8')
    test_file.unlink()
    print(f'[OK] Output directory writable: {out_dir.resolve()}')
except Exception as e:
    print(f'[FAIL] Cannot write to output dir: {e}')

# Print active settings summary
print(f'\nActive settings:')
print(f'  Problem     : {cfg["problem"]["name"]} ({cfg["problem"]["physics"]})')
print(f'  Element     : {cfg["elements"]["type"]}')
print(f'  Solver type : {cfg["solver"]["type"]}  (nlgeom={cfg["solver"]["nlgeom"]})')
print(f'  MAPDL port  : {cfg["mapdl"]["port"]}')
print(f'  RAM budget  : {cfg["mapdl"].get("ram_mb", "auto")} MB')

<a id='2'></a>
## Stage 2 — Geometry Import Verification

### Common nTopology CDB import issues

| Problem | Symptom | Fix |
|---------|---------|-----|
| SOLID185 for thin sheet | Too stiff in bending, unphysical results | `gi.reassign_element_type('SHELL181', ...)` |
| Node CS not aligned | Wrong DOF direction with D,ALL,UX | `mapdl.nrotat('ALL')` — done by default |
| Missing section thickness | SHELL elements have no real constants | `mapdl.sectype(1,'SHELL'); mapdl.secdata(t)` |
| Old nTop version ETYPE | Elements assigned wrong type | Check `mapdl.run('ETLIST')` before solving |

In [ ]:
# This cell runs only if you have a live mapdl instance.
# Replace 'mapdl' with your actual connection.

# from ams.mapdl.runner import MAPDLRunner
# runner = MAPDLRunner(cfg)
# mapdl  = runner.connect()

# --- geometry import debug checklist ---
# 1. Import the CDB
# from ams.geometry.importer import GeometryImporter
# gi = GeometryImporter(mapdl)
# gi.from_cdb(cfg['geometry']['cdb_path'])

# 2. Print element type list -- are SOLID185s present?
# print(mapdl.run('ETLIST'))

# 3. Print mesh statistics
# print(f'Nodes    : {mapdl.mesh.n_node:,}')
# print(f'Elements : {mapdl.mesh.n_elem:,}')

# 4. If SOLID185 is present for a shell model, reassign:
# gi.reassign_element_type('SHELL181', section_thickness_m=0.001)

print('Uncomment the cells above after connecting to MAPDL.')
print('See notebooks/03_geometry_import_and_mesh_quality.ipynb for the full workflow.')

<a id='3'></a>
## Stage 3 — Mesh Quality Gate

**The mesh quality checker blocks the solve if any critical check fails.**
Always run this before `run_solution()`.

In [ ]:
# With a live mapdl instance:
# from ams.geometry.mesh_quality import MeshQualityChecker, QualityThresholds
# checker = MeshQualityChecker(mapdl)
# report  = checker.check(raise_on_fail=False)   # set True to block the solve on failure
# report.print_summary()

# What to look for:
# [X] jacobian <= 0   -> inverted elements -> fix mesh (nTop geometry, STL winding)
# [!] aspect_ratio > 20 -> ill-conditioned -> refine mesh near singularities
# [!] warping > 30deg  -> shell quality    -> reduce mesh element size at creases
# [!] not manifold     -> geometry error   -> fix CAD before re-exporting CDB

# Offline: test the standalone geometry check functions (no MAPDL needed)
import numpy as np
from ams.geometry.mesh_quality import _estimate_jacobian

test_cases = [
    (np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0]], dtype=float), 'CCW quad (good)'),
    (np.array([[0,0,0],[0,1,0],[1,1,0],[1,0,0]], dtype=float), 'CW quad (inverted)'),
]
for coords, name in test_cases:
    j = _estimate_jacobian(coords)
    flag = '[OK]' if j > 0 else '[INVERTED - fatal]'
    print(f'  {flag}  {name}: J = {j:.4f}')

<a id='4'></a>
## Stage 4 — BC Sanity Check

### Rigid body mode checklist

The most common BC mistake is under-constraining the model — leaving
rigid body modes that make [K] singular.

**2D (plane stress/strain):** need to remove 3 DOFs (2 translation + 1 rotation)
- Fix UX on left face: removes TX
- Fix UY at one node on left face: removes TY and RZ (single-point pin)

**3D solid:** need to remove 6 DOFs (3 translation + 3 rotation)
- Fix a face: removes translation perpendicular to face + 2 in-plane rotations
- Fix an edge: removes the remaining DOFs

### Pressure sign convention

ANSYS: positive pressure = compressive.  For tensile traction:
```python
# WRONG: this applies 100 MPa compressive (pushes the face inward)
mapdl.sf('ALL', 'PRES', 100e6)

# CORRECT: negative = tensile (pulls the face outward)
mapdl.sf('ALL', 'PRES', -100e6)
# Or use the toolbox:
# apply_neumann_pressure(mapdl, 'face', 'X', x_max, pressure_Pa=-100e6)
```

In [ ]:
# BC checklist (offline — verify config)
import yaml
from pathlib import Path

with open(Path('..') / 'config.yaml', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

bcs = cfg.get('bcs', {})
constraints = bcs.get('constraints', [])
loads       = bcs.get('loads',       [])

print(f'Constraints: {len(constraints)}')
for i, bc in enumerate(constraints):
    print(f'  {i+1}. type={bc.get("type")}  axis={bc.get("axis")}  side={bc.get("side")}  dofs={bc.get("dofs","ALL")}')

print(f'Loads: {len(loads)}')
for i, bc in enumerate(loads):
    if bc.get('type') == 'pressure':
        val = bc.get('value_Pa', 0)
        sign_note = 'tensile (correct for tension)' if val < 0 else 'compressive'
        print(f'  {i+1}. pressure={val:.2e} Pa ({sign_note})')
    else:
        print(f'  {i+1}. type={bc.get("type")}')

# Rigid body mode check (2D)
physics = cfg['problem'].get('physics', 'structural')
dim     = cfg['problem'].get('spatial_dim', 3)
n_rbm   = 3 if dim == 2 else 6

fixed_dofs = set()
for bc in constraints:
    dofs = bc.get('dofs', [])
    if isinstance(dofs, list):
        fixed_dofs.update(d.upper() for d in dofs)
    elif isinstance(dofs, str) and dofs.upper() == 'ALL':
        fixed_dofs.update(['UX', 'UY', 'UZ', 'ROTX', 'ROTY', 'ROTZ'])

print(f'\nFixed DOFs: {sorted(fixed_dofs)}')
print(f'(Need at least {n_rbm} independent constraints for a {dim}D model)')

if 'UX' not in fixed_dofs and 'ALL' not in fixed_dofs:
    print('[!] UX not fixed -- model may have a rigid-body translation mode along X')

<a id='5'></a>
## Stage 5 — Solver Diagnostic Run

### Interpreting divergence

| Symptom in MAPDL log | Root cause | Fix |
|----------------------|------------|-----|
| `*** WARNING *** MAXIMUM NUMBER OF EQUILIBRIUM ITERATIONS REACHED` | NR failed to converge | Increase `nsubst.max`, check mesh quality |
| `*** ERROR *** ELEMENT ... IS DISTORTED` | Element distorted during deformation | Enable `nlgeom: true`, reduce load step |
| `*** ERROR *** NEGATIVE JACOBIAN AT ELEMENT ...` | Inverted element in mesh | Fix geometry, check CDB import |
| `*** WARNING *** PIVOT` | Near-singular stiffness | Check BCs (rigid body mode), check material |
| Residual plateaus (doesn't decrease) | BC error or mesh locking | Check element type, integration scheme |
| Residual oscillates | Load step too large | Reduce initial substeps, enable LNSRCH |

### Reading the MAPDL solve log

In [ ]:
# Parse MAPDL .log or .out file for error/warning lines
from pathlib import Path
import re

def scan_mapdl_log(log_path: str) -> dict:
    """Scan a MAPDL .log or .out file for errors, warnings, and convergence info."""
    path = Path(log_path)
    if not path.exists():
        return {'error': f'File not found: {path}'}

    errors   = []
    warnings = []
    converged_substeps = []
    diverged_substeps  = []

    with open(path, encoding='utf-8', errors='replace') as fh:
        for lineno, line in enumerate(fh, 1):
            if re.search(r'\*\*\* ERROR', line):
                errors.append((lineno, line.strip()))
            elif re.search(r'\*\*\* WARNING', line):
                warnings.append((lineno, line.strip()))
            elif re.search(r'CONVERGENCE', line, re.IGNORECASE):
                if 'NOT' in line.upper() or 'FAIL' in line.upper():
                    diverged_substeps.append((lineno, line.strip()))
                else:
                    converged_substeps.append((lineno, line.strip()))

    return {
        'errors':           errors,
        'warnings':         warnings,
        'converged_count':  len(converged_substeps),
        'diverged_count':   len(diverged_substeps),
    }


# Try to find any MAPDL log files
import yaml
with open('../config.yaml', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

run_loc = cfg['mapdl'].get('run_location') or 'C:/Users'
jobname = cfg['mapdl'].get('jobname', 'ams_run')
log_path = Path(run_loc) / f'{jobname}.out'

# Also check project root for stale .err files
import glob
log_candidates = (
    list(Path('..').glob('file*.out')) +
    list(Path('..').glob('file*.err')) +
    [log_path]
)
existing = [p for p in log_candidates if p.exists()]

if existing:
    for f in existing[:3]:
        print(f'\nScanning: {f}')
        result = scan_mapdl_log(str(f))
        print(f'  Errors   : {len(result["errors"])}')
        print(f'  Warnings : {len(result["warnings"])}')
        print(f'  Converged substeps : {result["converged_count"]}')
        print(f'  Diverged substeps  : {result["diverged_count"]}')
        for lineno, msg in result['errors'][:5]:
            print(f'  [ERROR L{lineno}] {msg[:100]}')
        for lineno, msg in result['warnings'][:5]:
            print(f'  [WARN  L{lineno}] {msg[:100]}')
else:
    print('No MAPDL log files found.  Run a simulation first.')
    print('Log files are written to: mapdl.run_location (or OS temp dir)')

<a id='6'></a>
## Stage 6 — Results Sanity Check

### Quick checks after a solve

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find the latest nodal_results.csv
csv_files = sorted(
    Path('../outputs').rglob('nodal_results.csv'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not csv_files:
    print('No results CSV found.  Run a simulation first.')
else:
    csv = csv_files[0]
    df  = pd.read_csv(csv)
    print(f'Latest results: {csv}')
    print(f'Shape: {df.shape[0]} nodes x {df.shape[1]-1} quantities')
    print()

    for col in df.columns:
        if col == 'node_index': continue
        vals = df[col].dropna()
        n_nan = df[col].isna().sum()
        n_inf = np.isinf(vals).sum()
        flag  = ''
        if n_nan > 0:               flag += f' [!] {n_nan} NaN values'
        if n_inf > 0:               flag += f' [!] {n_inf} Inf values'
        if vals.max() == vals.min():flag += ' [!] all values identical (solve may have failed)'
        print(f'  {col:<30} min={vals.min():+.4g}  max={vals.max():+.4g}{flag}')

    # Quick plot
    numeric_cols = [c for c in df.columns if c != 'node_index']
    if numeric_cols:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(df[numeric_cols[0]], 'b.', markersize=2)
        ax.set_title(f'{numeric_cols[0]} — per-node distribution')
        ax.set_xlabel('Node index'); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

### Expected magnitudes — quick reference

| Quantity | Typical range | If wrong, check |
|----------|--------------|----------------|
| `displacement_norm` | 0.001–0.1 × characteristic_length | BC sign, load magnitude |
| `von_mises` | 0–5× σ_yield | Material model, BC sign |
| `displacement_x` | symmetric about 0 for symmetric geometry | Symmetry BC |
| All zeros | Solve didn't run | Check `.out` log for errors |
| All same value | POST1 read wrong result set | `mapdl.set('LAST')` |

<a id='7'></a>
## Stage 7 — Port / Process Audit

Run this when MAPDL hangs on connection or license checkout.

In [ ]:
import subprocess, sys, platform

# Check ANSYS license file environment variable
import os
license_file = os.environ.get('ANSYSLMD_LICENSE_FILE', '')
print(f'ANSYSLMD_LICENSE_FILE = {license_file or "(not set)"}')
if not license_file:
    print('[!] License variable not set.  MAPDL will look for a default license server.')
    print('    Set it if your license server is not on localhost:')
    print('    e.g., set ANSYSLMD_LICENSE_FILE=1055@your-license-server')

# List any TCP connections to common ANSYS ports
print(f'\nTCP port usage (relevant ANSYS ports):')
if platform.system() == 'Windows':
    try:
        result = subprocess.run(
            ['netstat', '-an'],
            capture_output=True, text=True, timeout=10
        )
        for line in result.stdout.splitlines():
            if any(str(p) in line for p in [50052, 50051, 50053, 50054, 1055, 2325]):
                print(f'  {line.strip()}')
    except Exception as e:
        print(f'  Could not run netstat: {e}')
else:
    print('  (netstat check is Windows-specific)')

print()
print('Common ANSYS port assignments:')
print('  1055  -- FlexNet license daemon (ansys-flexlm)')
print('  2325  -- ANSYS license server')
print('  50051 -- PyAEDT gRPC (HFSS)')
print('  50052 -- PyMAPDL gRPC (MAPDL, default)')
print('  50053+ - Additional MAPDL instances')

<a id='8'></a>
## Stage 8 — HFSS-Specific Debug Checklist

From real bugs documented in the origami EMI workflow:

In [ ]:
from pathlib import Path

# HFSS project file audit
import yaml
with open('../config.yaml', encoding='utf-8') as fh:
    cfg = yaml.safe_load(fh)

hfss_cfg     = cfg.get('hfss', {})
project_name = hfss_cfg.get('project_name', 'ams_hfss')
project_dir  = Path(cfg['output']['dir']) / 'hfss_project'

print('HFSS Project File Audit')
print(f'  Project dir  : {project_dir}')
print(f'  Project name : {project_name}')
print()

stale_files = [
    project_dir / f'{project_name}.aedt',
    project_dir / f'{project_name}.aedt.auto',
    project_dir / f'{project_name}.lock',
    project_dir / f'{project_name}.aedtresults',
]

for p in stale_files:
    if p.exists():
        size = p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        print(f'  [FOUND] {p.name} ({size/1024:.1f} KB) -- may cause launch issues')
    else:
        print(f'  [OK   ] {p.name} (not present)')

print()
print('To clean these before the next HFSS launch:')
print('  from ams.hfss.runner import cleanup_hfss_project')
print(f'  cleanup_hfss_project(Path("{project_dir}"), "{project_name}")')
print()
print('HFSS debug checklist:')
checklist = [
    'non_graphical=True is set in config.yaml (hfss.non_graphical)',
    'new_desktop=True is used in HFSSRunner (always opens fresh session)',
    'cleanup_hfss_project() called before launch',
    'STL imported with import_free_surfaces=True for thin sheets',
    '_Unnamed_6 objects deleted after STL import',
    'Floquet ports assigned to BOTH top and bottom air box faces',
    'Periodic master-slave on BOTH X and Y faces (2D periodic array)',
    'Air box clearance >= lambda/4 at highest frequency',
    'aedt_version in config matches installed AEDT version exactly',
]
for item in checklist:
    print(f'  [ ] {item}')

---

## Quick Reference: Error Lookup

```
OSError: [WinError 10048] Only one usage of each socket address
  -> Port 50052 occupied. Run kill_ansys_zombies() then check_ports().

RuntimeError: Could not connect to MAPDL on port 50052
  -> Same as above, or ANSYS binary path wrong. Check mapdl.custom_bin in config.

*** ERROR *** NEGATIVE JACOBIAN AT ELEMENT XXX
  -> Inverted element in the mesh. Fix geometry before export, or
     check nTop CDB for non-manifold faces.

*** WARNING *** MAXIMUM NUMBER OF EQUILIBRIUM ITERATIONS REACHED
  -> NR failed to converge. Increase nsubst.max, check BCs, enable LNSRCH.

HFSS: setup already exists / DrivenModal registration failed
  -> Stale .aedtresults directory. Run cleanup_hfss_project() first.

HFSS: meshing failed / non-manifold geometry
  -> Delete _Unnamed_6 objects after STL import with import_free_surfaces=True.

NaN in results CSV
  -> Solve diverged. Check .out log for *** ERROR lines.

von Mises stress = 0 everywhere
  -> Likely read result set 0 instead of LAST. Call mapdl.set('LAST') in POST1.

displacement = 0 at prescribed DOF (should be UX=0.1)
  -> BC applied to wrong node set. Check nsel + D commands in prep7.
```

**If none of the above match:** check `ANSYS_SIM_TOOLBOX_GUIDE.md §14 Bug Catalogue`.